# Graph convolutional network

A seed-42 rerun that produces an out-of-fold prediction for each genome. Requires the study graph files, fold manifest, model checkpoint, and reference fold results. Set PROJECT_ROOT to a working copy of the prepared study inputs. This notebook writes results under that directory.

In [ ]:
# ================================================================
# CELL 3 — SEED-42 GCN 10-FOLD OOF RERUN
#
# Recommended configuration:
#   NVIDIA A100 GPU
#
# Expected runtime:
#   Approximately 3–8 minutes
#
# Purpose:
#   - Rerun only GCN seed 42.
#   - Use the exact reconstructed 2,448-graph cohort.
#   - Train one fresh model per fold.
#   - Generate one held-out prediction per genome.
#   - Save probabilities, predictions, metrics, fold results,
#     training histories, and fold-specific checkpoints.
#
# This cell does NOT generate Supplementary Figures S1 and S2 yet.
# ================================================================

In [ ]:
# ================================================================
# 0. IMPORTS
# ================================================================

import os
import gc
import json
import time
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Subset

from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    GCNConv,
    global_mean_pool,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [ ]:
# ================================================================
# 1. PATHS
# ================================================================

PROJECT_ROOT = Path(
    "project_data"
)

REVISION_ROOT = (
    PROJECT_ROOT
    / "Frontiers_Revision_Outputs"
)

TABLES_DIR = (
    REVISION_ROOT
    / "tables"
)

ROBUSTNESS_DIR = (
    TABLES_DIR
    / "gcn_seed_robustness"
)

MODEL_DIR = (
    REVISION_ROOT
    / "models"
    / "gcn_seed_robustness"
)

OUTPUT_DIR = (
    TABLES_DIR
    / "gcn_seed42_2448_supplementary_figures"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH = (
    OUTPUT_DIR
    / "gcn_seed42_2448_rerun_manifest.csv"
)

REFERENCE_FOLD_PATH = (
    ROBUSTNESS_DIR
    / "gcn_seed_robustness_per_fold.csv"
)

FULL_MODEL_PATH = (
    MODEL_DIR
    / "gcn_full_data_seed_42.pt"
)

FOLD_MODEL_DIR = (
    OUTPUT_DIR
    / "seed42_fold_models"
)

FOLD_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OOF_PATH = (
    OUTPUT_DIR
    / "gcn_seed42_2448_oof_predictions.csv"
)

METRICS_PATH = (
    OUTPUT_DIR
    / "gcn_seed42_2448_metrics.csv"
)

PER_FOLD_PATH = (
    OUTPUT_DIR
    / "gcn_seed42_2448_per_fold_metrics.csv"
)

FOLD_COMPARISON_PATH = (
    OUTPUT_DIR
    / "gcn_seed42_2448_saved_vs_rerun_fold_comparison.csv"
)

TRAINING_HISTORY_PATH = (
    OUTPUT_DIR
    / "gcn_seed42_2448_training_history.csv"
)

RUN_METADATA_PATH = (
    OUTPUT_DIR
    / "gcn_seed42_2448_run_metadata.json"
)

CHECKPOINT_OOF_PATH = (
    OUTPUT_DIR
    / "CHECKPOINT_gcn_seed42_2448_partial_oof.csv"
)

CHECKPOINT_FOLD_PATH = (
    OUTPUT_DIR
    / "CHECKPOINT_gcn_seed42_2448_completed_folds.csv"
)

In [ ]:
# ================================================================
# 2. FIXED SETTINGS
# ================================================================

SEED = 42
N_SPLITS = 10

LEARNING_RATE = 0.001
EPOCHS = 20
BATCH_SIZE = 16

NUM_EMBEDDINGS = 10000
EMBEDDING_DIM = 128
HIDDEN_CHANNELS = 128

PROBABILITY_THRESHOLD = 0.5

EXPECTED_N = 2448
EXPECTED_SUSCEPTIBLE = 1815
EXPECTED_RESISTANT = 633

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
# ================================================================
# 3. REQUIRED FILE AND GPU CHECKS
# ================================================================

required_paths = [
    MANIFEST_PATH,
    REFERENCE_FOLD_PATH,
    FULL_MODEL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required files are missing. Cell 2 may need to be "
        "rerun in the current Colab session:\n"
        + "\n".join(missing_paths)
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is not active. Switch Colab to a GPU "
        "runtime before beginning this rerun."
    )

print("=" * 80)
print("SEED-42 GCN OOF RERUN")
print("=" * 80)

print("Device:", DEVICE)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)
print("PyTorch:", torch.__version__)

try:
    import torch_geometric

    print(
        "PyTorch Geometric:",
        torch_geometric.__version__,
    )
except Exception:
    pass

In [ ]:
# ================================================================
# 4. REPRODUCIBILITY FUNCTION
# ================================================================

def set_all_seeds(seed):
    """
    Set Python, NumPy, CPU PyTorch, and CUDA seeds.

    The same model seed is reset before every fold, while the
    held-out partition remains the predefined seed-42 split.
    """

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
# ================================================================
# 5. MODEL ARCHITECTURE
# ================================================================

class GeneGCN(nn.Module):
    """
    Gene-adjacency graph GCN matching the saved seed-42
    checkpoint architecture.
    """

    def __init__(
        self,
        num_embeddings=NUM_EMBEDDINGS,
        embedding_dim=EMBEDDING_DIM,
        hidden_channels=HIDDEN_CHANNELS,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings,
            embedding_dim,
        )

        self.conv1 = GCNConv(
            embedding_dim,
            hidden_channels,
        )

        self.conv2 = GCNConv(
            hidden_channels,
            hidden_channels,
        )

        self.linear = nn.Linear(
            hidden_channels,
            1,
        )

    def forward(self, data):
        node_ids = (
            data.x
            .view(-1)
            .long()
        )

        x = self.embedding(
            node_ids
        )

        x = self.conv1(
            x,
            data.edge_index,
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            data.edge_index,
        )

        x = F.relu(x)

        x = global_mean_pool(
            x,
            data.batch,
        )

        logits = (
            self.linear(x)
            .view(-1)
        )

        return logits

In [ ]:
# ================================================================
# 6. VALIDATE ARCHITECTURE AGAINST SAVED CHECKPOINT
# ================================================================

saved_checkpoint = torch.load(
    FULL_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

if not isinstance(
    saved_checkpoint,
    dict,
):
    raise TypeError(
        "Unexpected saved checkpoint type: "
        f"{type(saved_checkpoint)}"
    )

if "model_state_dict" not in saved_checkpoint:
    raise KeyError(
        "Saved checkpoint does not contain model_state_dict."
    )

print("\nSaved checkpoint metadata:")

for key in [
    "seed",
    "num_embeddings",
    "embedding_dim",
    "hidden_channels",
    "learning_rate",
    "epochs",
    "batch_size",
    "n_graphs",
    "class_counts",
]:
    print(
        f"  {key}:",
        saved_checkpoint.get(
            key,
            "not recorded",
        ),
    )

checkpoint_expectations = {
    "num_embeddings": NUM_EMBEDDINGS,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_channels": HIDDEN_CHANNELS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "n_graphs": EXPECTED_N,
}

for key, expected_value in (
    checkpoint_expectations.items()
):
    recorded_value = saved_checkpoint.get(
        key
    )

    if (
        recorded_value is not None
        and int(recorded_value)
        != int(expected_value)
    ):
        raise ValueError(
            f"Checkpoint setting mismatch for {key}: "
            f"expected {expected_value}, "
            f"found {recorded_value}."
        )

recorded_lr = saved_checkpoint.get(
    "learning_rate"
)

if (
    recorded_lr is not None
    and not np.isclose(
        float(recorded_lr),
        LEARNING_RATE,
    )
):
    raise ValueError(
        "Checkpoint learning-rate mismatch: "
        f"{recorded_lr}"
    )

architecture_test_model = GeneGCN()

architecture_test_model.load_state_dict(
    saved_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

del architecture_test_model

print(
    "\nPASS: the reconstructed model architecture "
    "strictly loads the saved seed-42 state dictionary."
)

In [ ]:
# ================================================================
# 7. LOAD AND VALIDATE THE FINAL MANIFEST
# ================================================================

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "source_file": str,
        "graph_path": str,
        "genome_id": str,
        "genome_name": str,
        "true_phenotype": str,
    },
)

required_columns = {
    "graph_index",
    "source_file",
    "graph_path",
    "genome_id",
    "genome_name",
    "true_label",
    "true_phenotype",
    "fold",
    "seed",
}

if not required_columns.issubset(
    manifest.columns
):
    raise ValueError(
        "Rerun manifest is missing required columns.\n"
        f"Found: {manifest.columns.tolist()}"
    )

manifest = (
    manifest
    .sort_values("graph_index")
    .reset_index(drop=True)
)

manifest[
    "graph_index"
] = pd.to_numeric(
    manifest["graph_index"],
    errors="raise",
).astype(int)

manifest[
    "true_label"
] = pd.to_numeric(
    manifest["true_label"],
    errors="raise",
).astype(int)

manifest[
    "fold"
] = pd.to_numeric(
    manifest["fold"],
    errors="raise",
).astype(int)

if len(manifest) != EXPECTED_N:
    raise ValueError(
        f"Expected {EXPECTED_N} manifest rows, "
        f"found {len(manifest)}."
    )

if manifest[
    "graph_index"
].tolist() != list(
    range(EXPECTED_N)
):
    raise ValueError(
        "Manifest graph_index values are not exactly 0–2447."
    )

if manifest[
    "genome_id"
].nunique() != EXPECTED_N:
    raise ValueError(
        "Genome IDs are not unique."
    )

if manifest[
    "source_file"
].nunique() != EXPECTED_N:
    raise ValueError(
        "Graph filenames are not unique."
    )

class_counts = (
    manifest[
        "true_label"
    ]
    .value_counts()
    .to_dict()
)

if class_counts != {
    0: EXPECTED_SUSCEPTIBLE,
    1: EXPECTED_RESISTANT,
}:
    raise ValueError(
        "Unexpected class counts: "
        f"{class_counts}"
    )

fold_counts = (
    manifest["fold"]
    .value_counts()
    .sort_index()
)

if set(
    fold_counts.index.tolist()
) != set(range(1, 11)):
    raise ValueError(
        "Manifest does not contain folds 1–10."
    )

print("\nManifest validation:")

print("  Total N:", len(manifest))
print(
    "  Unique genome IDs:",
    manifest["genome_id"].nunique(),
)
print(
    "  Susceptible:",
    int(
        (
            manifest["true_label"]
            == 0
        ).sum()
    ),
)
print(
    "  Resistant:",
    int(
        (
            manifest["true_label"]
            == 1
        ).sum()
    ),
)

print("\nFold counts:")
print(
    fold_counts.to_string()
)

In [ ]:
# ================================================================
# 8. PRELOAD ALL GRAPH OBJECTS
# ================================================================

print("\n" + "=" * 80)
print("PRELOADING 2,448 GRAPH OBJECTS")
print("=" * 80)

graphs = []

for row_number, row in (
    manifest.iterrows()
):
    graph_path = Path(
        row["graph_path"]
    )

    if not graph_path.exists():
        raise FileNotFoundError(
            f"Graph file is missing: {graph_path}\n"
            "Cell 2 may need to be rerun in this runtime."
        )

    graph = torch.load(
        graph_path,
        map_location="cpu",
        weights_only=False,
    )

    graph.x = (
        graph.x
        .view(-1)
        .long()
    )

    graph.edge_index = (
        graph.edge_index
        .long()
    )

    # Replace the stored y tensor with the independently
    # verified phenotype label from the final manifest.
    graph.y = torch.tensor(
        [
            float(
                row["true_label"]
            )
        ],
        dtype=torch.float32,
    )

    # Avoid a field name containing "index", because PyG may
    # automatically increment index-like attributes when batching.
    graph.record_id = torch.tensor(
        [
            int(
                row["graph_index"]
            )
        ],
        dtype=torch.long,
    )

    graphs.append(graph)

    if (
        row_number + 1
    ) % 250 == 0:
        print(
            f"  Loaded "
            f"{row_number + 1}/{EXPECTED_N}"
        )

if len(graphs) != EXPECTED_N:
    raise RuntimeError(
        "Not all graphs were loaded."
    )

print(
    "\nPASS: all 2,448 graphs are loaded in the "
    "original robustness-manifest order."
)

In [ ]:
# ================================================================
# 9. METRIC FUNCTION
# ================================================================

def calculate_binary_metrics(
    true_labels,
    probabilities,
    predictions,
):
    true_labels = np.asarray(
        true_labels,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    predictions = np.asarray(
        predictions,
        dtype=int,
    )

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "N": int(
            len(true_labels)
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Accuracy": float(
            accuracy_score(
                true_labels,
                predictions,
            )
        ),
        "Balanced accuracy": float(
            balanced_accuracy_score(
                true_labels,
                predictions,
            )
        ),
        "ROC-AUC": float(
            roc_auc_score(
                true_labels,
                probabilities,
            )
        ),
        "Precision": float(
            precision_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),
        "Recall/Sensitivity": float(
            recall_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),
        "Specificity": float(
            specificity
        ),
        "F1": float(
            f1_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),
    }

In [ ]:
# ================================================================
# 10. OOF STORAGE
# ================================================================

oof_logits = np.full(
    EXPECTED_N,
    np.nan,
    dtype=np.float64,
)

oof_probabilities = np.full(
    EXPECTED_N,
    np.nan,
    dtype=np.float64,
)

oof_predictions = np.full(
    EXPECTED_N,
    -1,
    dtype=np.int64,
)

prediction_counts = np.zeros(
    EXPECTED_N,
    dtype=np.int64,
)

fold_metric_rows = []
training_history_rows = []

overall_start = time.perf_counter()

In [ ]:
# ================================================================
# 11. TRAIN AND EVALUATE EACH FOLD
# ================================================================

for fold_number in range(
    1,
    N_SPLITS + 1,
):
    fold_start = time.perf_counter()

    print("\n" + "=" * 80)
    print(
        f"FOLD {fold_number}/{N_SPLITS}"
    )
    print("=" * 80)

    set_all_seeds(SEED)

    test_indices = (
        manifest.loc[
            manifest["fold"]
            == fold_number,
            "graph_index",
        ]
        .to_numpy(dtype=int)
    )

    train_indices = (
        manifest.loc[
            manifest["fold"]
            != fold_number,
            "graph_index",
        ]
        .to_numpy(dtype=int)
    )

    if np.intersect1d(
        train_indices,
        test_indices,
    ).size != 0:
        raise RuntimeError(
            f"Train/test overlap detected in fold {fold_number}."
        )

    if (
        len(train_indices)
        + len(test_indices)
        != EXPECTED_N
    ):
        raise RuntimeError(
            f"Incomplete fold partition in fold {fold_number}."
        )

    train_dataset = Subset(
        graphs,
        train_indices.tolist(),
    )

    test_dataset = Subset(
        graphs,
        test_indices.tolist(),
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    model = GeneGCN(
        num_embeddings=NUM_EMBEDDINGS,
        embedding_dim=EMBEDDING_DIM,
        hidden_channels=HIDDEN_CHANNELS,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0,
        amsgrad=False,
    )

    # The original documented protocol used ordinary,
    # unweighted BCEWithLogitsLoss.
    criterion = nn.BCEWithLogitsLoss()

    final_train_loss = np.nan

    # ------------------------------------------------------------
    # Train for exactly 20 epochs
    # ------------------------------------------------------------

    for epoch_number in range(
        1,
        EPOCHS + 1,
    ):
        model.train()

        total_weighted_loss = 0.0
        total_training_graphs = 0

        for batch in train_loader:
            batch = batch.to(
                DEVICE
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(batch)

            targets = (
                batch.y
                .view(-1)
                .float()
            )

            loss = criterion(
                logits,
                targets,
            )

            loss.backward()

            optimizer.step()

            number_graphs = int(
                batch.num_graphs
            )

            total_weighted_loss += (
                float(
                    loss.detach()
                    .cpu()
                    .item()
                )
                * number_graphs
            )

            total_training_graphs += (
                number_graphs
            )

        epoch_train_loss = (
            total_weighted_loss
            / total_training_graphs
        )

        final_train_loss = (
            epoch_train_loss
        )

        training_history_rows.append(
            {
                "seed": SEED,
                "fold": fold_number,
                "epoch": epoch_number,
                "train_loss": (
                    epoch_train_loss
                ),
                "train_size": (
                    len(train_indices)
                ),
            }
        )

    # ------------------------------------------------------------
    # Held-out fold evaluation
    # ------------------------------------------------------------

    model.eval()

    fold_record_ids = []
    fold_true_labels = []
    fold_logits = []
    fold_probabilities = []
    fold_predictions = []

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(
                DEVICE
            )

            logits = model(batch)

            probabilities = torch.sigmoid(
                logits
            )

            # Preserve the documented original rule:
            # resistant when output logit > 0.
            predictions = (
                logits > 0
            ).long()

            record_ids = (
                batch.record_id
                .view(-1)
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )

            true_labels = (
                batch.y
                .view(-1)
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )

            fold_record_ids.extend(
                record_ids.tolist()
            )

            fold_true_labels.extend(
                true_labels.tolist()
            )

            fold_logits.extend(
                logits
                .detach()
                .cpu()
                .numpy()
                .astype(float)
                .tolist()
            )

            fold_probabilities.extend(
                probabilities
                .detach()
                .cpu()
                .numpy()
                .astype(float)
                .tolist()
            )

            fold_predictions.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
                .astype(int)
                .tolist()
            )

    fold_record_ids = np.asarray(
        fold_record_ids,
        dtype=int,
    )

    fold_true_labels = np.asarray(
        fold_true_labels,
        dtype=int,
    )

    fold_logits = np.asarray(
        fold_logits,
        dtype=float,
    )

    fold_probabilities = np.asarray(
        fold_probabilities,
        dtype=float,
    )

    fold_predictions = np.asarray(
        fold_predictions,
        dtype=int,
    )

    if len(fold_record_ids) != len(
        test_indices
    ):
        raise RuntimeError(
            f"Fold {fold_number} produced "
            f"{len(fold_record_ids)} predictions for "
            f"{len(test_indices)} held-out genomes."
        )

    if len(
        np.unique(
            fold_record_ids
        )
    ) != len(
        fold_record_ids
    ):
        raise RuntimeError(
            f"Duplicate genome predictions occurred "
            f"inside fold {fold_number}."
        )

    if set(
        fold_record_ids.tolist()
    ) != set(
        test_indices.tolist()
    ):
        raise RuntimeError(
            f"Fold {fold_number} prediction IDs do not "
            "match the predefined held-out set."
        )

    expected_fold_labels = (
        manifest.set_index(
            "graph_index"
        )
        .loc[
            fold_record_ids,
            "true_label",
        ]
        .to_numpy(dtype=int)
    )

    if not np.array_equal(
        fold_true_labels,
        expected_fold_labels,
    ):
        raise RuntimeError(
            f"Graph labels disagree with manifest labels "
            f"in fold {fold_number}."
        )

    # ------------------------------------------------------------
    # Store OOF predictions by permanent graph index
    # ------------------------------------------------------------

    for local_position, record_id in enumerate(
        fold_record_ids
    ):
        if prediction_counts[
            record_id
        ] != 0:
            raise RuntimeError(
                f"Genome row {record_id} already has an "
                "OOF prediction."
            )

        prediction_counts[
            record_id
        ] += 1

        oof_logits[
            record_id
        ] = fold_logits[
            local_position
        ]

        oof_probabilities[
            record_id
        ] = fold_probabilities[
            local_position
        ]

        oof_predictions[
            record_id
        ] = fold_predictions[
            local_position
        ]

    # ------------------------------------------------------------
    # Calculate held-out fold metrics
    # ------------------------------------------------------------

    fold_metrics = (
        calculate_binary_metrics(
            fold_true_labels,
            fold_probabilities,
            fold_predictions,
        )
    )

    fold_runtime_minutes = (
        time.perf_counter()
        - fold_start
    ) / 60

    fold_metric_row = {
        "Seed": SEED,
        "Fold": fold_number,
        "Learning rate": (
            LEARNING_RATE
        ),
        "Hidden channels": (
            HIDDEN_CHANNELS
        ),
        "Embedding dimension": (
            EMBEDDING_DIM
        ),
        "Epochs": EPOCHS,
        "Batch size": BATCH_SIZE,
        "Train size": (
            len(train_indices)
        ),
        "Test size": (
            len(test_indices)
        ),
        "Final train loss": (
            final_train_loss
        ),
        "Runtime minutes": (
            fold_runtime_minutes
        ),
        **fold_metrics,
    }

    fold_metric_rows.append(
        fold_metric_row
    )

    print(
        f"Train size: {len(train_indices)}"
    )

    print(
        f"Test size: {len(test_indices)}"
    )

    print(
        f"Final train loss: "
        f"{final_train_loss:.6f}"
    )

    print(
        f"TN={fold_metrics['TN']}, "
        f"FP={fold_metrics['FP']}, "
        f"FN={fold_metrics['FN']}, "
        f"TP={fold_metrics['TP']}"
    )

    print(
        f"Accuracy="
        f"{fold_metrics['Accuracy']:.6f}"
    )

    print(
        f"ROC-AUC="
        f"{fold_metrics['ROC-AUC']:.6f}"
    )

    print(
        f"F1="
        f"{fold_metrics['F1']:.6f}"
    )

    print(
        f"Fold runtime="
        f"{fold_runtime_minutes:.2f} minutes"
    )

    # ------------------------------------------------------------
    # Save fold-specific model checkpoint
    # ------------------------------------------------------------

    fold_model_path = (
        FOLD_MODEL_DIR
        / (
            f"gcn_seed42_fold_"
            f"{fold_number:02d}.pt"
        )
    )

    torch.save(
        {
            "seed": SEED,
            "fold": fold_number,
            "model_state_dict": (
                model.state_dict()
            ),
            "num_embeddings": (
                NUM_EMBEDDINGS
            ),
            "embedding_dim": (
                EMBEDDING_DIM
            ),
            "hidden_channels": (
                HIDDEN_CHANNELS
            ),
            "learning_rate": (
                LEARNING_RATE
            ),
            "epochs": EPOCHS,
            "batch_size": (
                BATCH_SIZE
            ),
            "loss_function": (
                "BCEWithLogitsLoss "
                "(unweighted)"
            ),
            "prediction_rule": (
                "resistant when logit > 0"
            ),
            "train_indices": (
                train_indices.tolist()
            ),
            "test_indices": (
                test_indices.tolist()
            ),
            "final_train_loss": (
                final_train_loss
            ),
            "fold_metrics": (
                fold_metrics
            ),
        },
        fold_model_path,
    )

    # ------------------------------------------------------------
    # Save resumable checkpoints after every fold
    # ------------------------------------------------------------

    partial_mask = (
        prediction_counts > 0
    )

    partial_oof = (
        manifest.loc[
            partial_mask,
            [
                "graph_index",
                "source_file",
                "genome_id",
                "genome_name",
                "true_label",
                "true_phenotype",
                "fold",
            ],
        ]
        .copy()
    )

    partial_indices = (
        partial_oof[
            "graph_index"
        ].to_numpy(dtype=int)
    )

    partial_oof[
        "predicted_logit_resistant"
    ] = oof_logits[
        partial_indices
    ]

    partial_oof[
        "predicted_probability_resistant"
    ] = oof_probabilities[
        partial_indices
    ]

    partial_oof[
        "predicted_label"
    ] = oof_predictions[
        partial_indices
    ]

    partial_oof[
        "seed"
    ] = SEED

    partial_oof[
        "prediction_count"
    ] = prediction_counts[
        partial_indices
    ]

    partial_oof.to_csv(
        CHECKPOINT_OOF_PATH,
        index=False,
    )

    pd.DataFrame(
        fold_metric_rows
    ).to_csv(
        CHECKPOINT_FOLD_PATH,
        index=False,
    )

    pd.DataFrame(
        training_history_rows
    ).to_csv(
        TRAINING_HISTORY_PATH,
        index=False,
    )

    # ------------------------------------------------------------
    # Release GPU memory before next fold
    # ------------------------------------------------------------

    del model
    del optimizer
    del criterion
    del train_loader
    del test_loader
    del train_dataset
    del test_dataset

    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ================================================================
# 12. COMPLETE OOF COVERAGE VALIDATION
# ================================================================

print("\n" + "=" * 80)
print("COMPLETE OOF COVERAGE VALIDATION")
print("=" * 80)

if not np.all(
    prediction_counts == 1
):
    count_summary = (
        pd.Series(
            prediction_counts
        )
        .value_counts()
        .sort_index()
    )

    raise RuntimeError(
        "Each genome did not receive exactly one prediction.\n"
        + count_summary.to_string()
    )

if np.isnan(
    oof_logits
).any():
    raise RuntimeError(
        "At least one OOF logit is missing."
    )

if np.isnan(
    oof_probabilities
).any():
    raise RuntimeError(
        "At least one OOF probability is missing."
    )

if np.any(
    oof_predictions < 0
):
    raise RuntimeError(
        "At least one OOF predicted label is missing."
    )

print(
    "Total predictions:",
    len(oof_predictions),
)

print(
    "Unique genomes:",
    manifest[
        "genome_id"
    ].nunique(),
)

print(
    "Predictions per genome:"
)

print(
    pd.Series(
        prediction_counts
    )
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "\nPASS: every genome received exactly one "
    "held-out prediction."
)

In [ ]:
# ================================================================
# 13. BUILD AND SAVE FINAL OOF TABLE
# ================================================================

oof_df = (
    manifest[
        [
            "graph_index",
            "source_file",
            "genome_id",
            "genome_name",
            "true_label",
            "true_phenotype",
            "fold",
        ]
    ]
    .copy()
)

oof_df[
    "predicted_logit_resistant"
] = oof_logits

oof_df[
    "predicted_probability_resistant"
] = oof_probabilities

oof_df[
    "predicted_label"
] = oof_predictions

oof_df[
    "predicted_phenotype"
] = (
    oof_df[
        "predicted_label"
    ]
    .map(
        {
            0: "Susceptible",
            1: "Resistant",
        }
    )
)

oof_df["seed"] = SEED

oof_df[
    "prediction_count"
] = prediction_counts

oof_df[
    "prediction_rule"
] = (
    "Resistant when output logit > 0; "
    "equivalent to sigmoid probability > 0.5"
)

oof_df.to_csv(
    OOF_PATH,
    index=False,
)

In [ ]:
# ================================================================
# 14. CALCULATE COMPLETE OOF METRICS
# ================================================================

true_labels = (
    oof_df[
        "true_label"
    ].to_numpy(dtype=int)
)

complete_oof_metrics = (
    calculate_binary_metrics(
        true_labels,
        oof_probabilities,
        oof_predictions,
    )
)

per_fold_df = pd.DataFrame(
    fold_metric_rows
)

per_fold_df.to_csv(
    PER_FOLD_PATH,
    index=False,
)

training_history_df = pd.DataFrame(
    training_history_rows
)

training_history_df.to_csv(
    TRAINING_HISTORY_PATH,
    index=False,
)

mean_fold_accuracy = float(
    per_fold_df[
        "Accuracy"
    ].mean()
)

sd_fold_accuracy = float(
    per_fold_df[
        "Accuracy"
    ].std(ddof=1)
)

mean_fold_auc = float(
    per_fold_df[
        "ROC-AUC"
    ].mean()
)

sd_fold_auc = float(
    per_fold_df[
        "ROC-AUC"
    ].std(ddof=1)
)

mean_fold_f1 = float(
    per_fold_df[
        "F1"
    ].mean()
)

sd_fold_f1 = float(
    per_fold_df[
        "F1"
    ].std(ddof=1)
)

In [ ]:
# ================================================================
# 15. COMPARE WITH SAVED SEED-42 REFERENCE
# ================================================================

saved_all_folds = pd.read_csv(
    REFERENCE_FOLD_PATH
)

saved_seed42 = (
    saved_all_folds[
        pd.to_numeric(
            saved_all_folds[
                "Seed"
            ],
            errors="coerce",
        ) == 42
    ]
    .sort_values("Fold")
    .reset_index(drop=True)
)

saved_tn = int(
    saved_seed42["TN"].sum()
)

saved_fp = int(
    saved_seed42["FP"].sum()
)

saved_fn = int(
    saved_seed42["FN"].sum()
)

saved_tp = int(
    saved_seed42["TP"].sum()
)

saved_pooled_accuracy = (
    saved_tn + saved_tp
) / EXPECTED_N

saved_mean_fold_accuracy = float(
    saved_seed42[
        "Accuracy"
    ].mean()
)

saved_mean_fold_auc = float(
    saved_seed42[
        "ROC-AUC"
    ].mean()
)

fold_comparison = (
    saved_seed42[
        [
            "Fold",
            "Final train loss",
            "TN",
            "FP",
            "FN",
            "TP",
            "Accuracy",
            "Balanced accuracy",
            "Precision",
            "Recall/Sensitivity",
            "Specificity",
            "F1",
            "ROC-AUC",
        ]
    ]
    .rename(
        columns={
            column: (
                f"saved_{column}"
            )
            for column in [
                "Final train loss",
                "TN",
                "FP",
                "FN",
                "TP",
                "Accuracy",
                "Balanced accuracy",
                "Precision",
                "Recall/Sensitivity",
                "Specificity",
                "F1",
                "ROC-AUC",
            ]
        }
    )
    .merge(
        per_fold_df[
            [
                "Fold",
                "Final train loss",
                "TN",
                "FP",
                "FN",
                "TP",
                "Accuracy",
                "Balanced accuracy",
                "Precision",
                "Recall/Sensitivity",
                "Specificity",
                "F1",
                "ROC-AUC",
            ]
        ].rename(
            columns={
                column: (
                    f"rerun_{column}"
                )
                for column in [
                    "Final train loss",
                    "TN",
                    "FP",
                    "FN",
                    "TP",
                    "Accuracy",
                    "Balanced accuracy",
                    "Precision",
                    "Recall/Sensitivity",
                    "Specificity",
                    "F1",
                    "ROC-AUC",
                ]
            }
        ),
        on="Fold",
        how="inner",
        validate="one_to_one",
    )
)

for metric in [
    "Final train loss",
    "TN",
    "FP",
    "FN",
    "TP",
    "Accuracy",
    "Balanced accuracy",
    "Precision",
    "Recall/Sensitivity",
    "Specificity",
    "F1",
    "ROC-AUC",
]:
    fold_comparison[
        f"difference_{metric}"
    ] = (
        fold_comparison[
            f"rerun_{metric}"
        ]
        - fold_comparison[
            f"saved_{metric}"
        ]
    )

fold_comparison.to_csv(
    FOLD_COMPARISON_PATH,
    index=False,
)

accuracy_difference = (
    complete_oof_metrics[
        "Accuracy"
    ]
    - saved_pooled_accuracy
)

mean_fold_accuracy_difference = (
    mean_fold_accuracy
    - saved_mean_fold_accuracy
)

mean_fold_auc_difference = (
    mean_fold_auc
    - saved_mean_fold_auc
)

confusion_matrix_exact_match = (
    complete_oof_metrics["TN"]
    == saved_tn
    and complete_oof_metrics["FP"]
    == saved_fp
    and complete_oof_metrics["FN"]
    == saved_fn
    and complete_oof_metrics["TP"]
    == saved_tp
)

meaningful_accuracy_discrepancy = (
    abs(
        accuracy_difference
    ) > 0.005
)

In [ ]:
# ================================================================
# 16. SAVE COMPLETE METRICS TABLE
# ================================================================

metrics_row = {
    "N": EXPECTED_N,
    "susceptible_count": (
        EXPECTED_SUSCEPTIBLE
    ),
    "resistant_count": (
        EXPECTED_RESISTANT
    ),
    "unique_genome_ids": int(
        oof_df[
            "genome_id"
        ].nunique()
    ),
    "accuracy": (
        complete_oof_metrics[
            "Accuracy"
        ]
    ),
    "balanced_accuracy": (
        complete_oof_metrics[
            "Balanced accuracy"
        ]
    ),
    "roc_auc_complete_oof": (
        complete_oof_metrics[
            "ROC-AUC"
        ]
    ),
    "precision": (
        complete_oof_metrics[
            "Precision"
        ]
    ),
    "recall_sensitivity": (
        complete_oof_metrics[
            "Recall/Sensitivity"
        ]
    ),
    "specificity": (
        complete_oof_metrics[
            "Specificity"
        ]
    ),
    "f1": (
        complete_oof_metrics[
            "F1"
        ]
    ),
    "TN": (
        complete_oof_metrics["TN"]
    ),
    "FP": (
        complete_oof_metrics["FP"]
    ),
    "FN": (
        complete_oof_metrics["FN"]
    ),
    "TP": (
        complete_oof_metrics["TP"]
    ),
    "threshold_probability": (
        PROBABILITY_THRESHOLD
    ),
    "prediction_rule": (
        "Resistant when output logit > 0; "
        "equivalent to sigmoid probability > 0.5"
    ),
    "seed": SEED,
    "number_of_folds": (
        N_SPLITS
    ),
    "learning_rate": (
        LEARNING_RATE
    ),
    "embedding_dimension": (
        EMBEDDING_DIM
    ),
    "hidden_channels": (
        HIDDEN_CHANNELS
    ),
    "epochs": EPOCHS,
    "batch_size": (
        BATCH_SIZE
    ),
    "loss_function": (
        "BCEWithLogitsLoss (unweighted)"
    ),
    "optimizer": (
        "Adam"
    ),
    "mean_fold_accuracy": (
        mean_fold_accuracy
    ),
    "fold_accuracy_sd": (
        sd_fold_accuracy
    ),
    "mean_fold_roc_auc": (
        mean_fold_auc
    ),
    "fold_roc_auc_sd": (
        sd_fold_auc
    ),
    "mean_fold_f1": (
        mean_fold_f1
    ),
    "fold_f1_sd": (
        sd_fold_f1
    ),
    "saved_reference_TN": (
        saved_tn
    ),
    "saved_reference_FP": (
        saved_fp
    ),
    "saved_reference_FN": (
        saved_fn
    ),
    "saved_reference_TP": (
        saved_tp
    ),
    "saved_reference_pooled_accuracy": (
        saved_pooled_accuracy
    ),
    "saved_reference_mean_fold_accuracy": (
        saved_mean_fold_accuracy
    ),
    "saved_reference_mean_fold_roc_auc": (
        saved_mean_fold_auc
    ),
    "pooled_accuracy_difference": (
        accuracy_difference
    ),
    "mean_fold_accuracy_difference": (
        mean_fold_accuracy_difference
    ),
    "mean_fold_roc_auc_difference": (
        mean_fold_auc_difference
    ),
    "confusion_matrix_exact_match": (
        confusion_matrix_exact_match
    ),
    "meaningful_accuracy_discrepancy_gt_0_005": (
        meaningful_accuracy_discrepancy
    ),
}

metrics_df = pd.DataFrame(
    [metrics_row]
)

metrics_df.to_csv(
    METRICS_PATH,
    index=False,
)

In [ ]:
# ================================================================
# 17. SAVE RUN METADATA
# ================================================================

total_runtime_minutes = (
    time.perf_counter()
    - overall_start
) / 60

run_metadata = {
    "generated_at": (
        pd.Timestamp.now().isoformat()
    ),
    "status": (
        "seed42_oof_rerun_complete"
    ),
    "device": str(
        DEVICE
    ),
    "gpu": (
        torch.cuda.get_device_name(0)
    ),
    "python_version": (
        platform.python_version()
    ),
    "pytorch_version": (
        torch.__version__
    ),
    "seed": SEED,
    "folds": N_SPLITS,
    "N": EXPECTED_N,
    "susceptible": (
        EXPECTED_SUSCEPTIBLE
    ),
    "resistant": (
        EXPECTED_RESISTANT
    ),
    "num_embeddings": (
        NUM_EMBEDDINGS
    ),
    "embedding_dimension": (
        EMBEDDING_DIM
    ),
    "hidden_channels": (
        HIDDEN_CHANNELS
    ),
    "learning_rate": (
        LEARNING_RATE
    ),
    "optimizer": (
        "Adam, betas=(0.9, 0.999), "
        "eps=1e-8, weight_decay=0"
    ),
    "loss": (
        "BCEWithLogitsLoss, unweighted"
    ),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "prediction_rule": (
        "resistant when logit > 0"
    ),
    "total_runtime_minutes": (
        total_runtime_minutes
    ),
    "output_files": {
        "oof_predictions": str(
            OOF_PATH
        ),
        "metrics": str(
            METRICS_PATH
        ),
        "per_fold_metrics": str(
            PER_FOLD_PATH
        ),
        "fold_comparison": str(
            FOLD_COMPARISON_PATH
        ),
        "training_history": str(
            TRAINING_HISTORY_PATH
        ),
        "fold_models_directory": str(
            FOLD_MODEL_DIR
        ),
    },
}

with open(
    RUN_METADATA_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        run_metadata,
        handle,
        indent=2,
    )

In [ ]:
# ================================================================
# 18. REQUIRED VALIDATION OUTPUT
# ================================================================

print("\n" + "=" * 80)
print("SEED-42 OOF RERUN COMPLETE")
print("=" * 80)

print("\nTotal N:")
print(EXPECTED_N)

print("\nClass counts:")
print(
    f"Susceptible: "
    f"{EXPECTED_SUSCEPTIBLE}"
)

print(
    f"Resistant: "
    f"{EXPECTED_RESISTANT}"
)

print("\nNumber of unique genome IDs:")
print(
    oof_df[
        "genome_id"
    ].nunique()
)

print("\nNumber of predictions per genome:")
print(
    oof_df[
        "prediction_count"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nFold counts:")
print(
    oof_df[
        "fold"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nConfusion matrix counts:")
print(
    f"TN={complete_oof_metrics['TN']}, "
    f"FP={complete_oof_metrics['FP']}, "
    f"FN={complete_oof_metrics['FN']}, "
    f"TP={complete_oof_metrics['TP']}"
)

print("\nComplete OOF metrics:")

print(
    f"Accuracy: "
    f"{complete_oof_metrics['Accuracy']:.8f}"
)

print(
    f"Balanced accuracy: "
    f"{complete_oof_metrics['Balanced accuracy']:.8f}"
)

print(
    f"ROC-AUC: "
    f"{complete_oof_metrics['ROC-AUC']:.8f}"
)

print(
    f"Precision: "
    f"{complete_oof_metrics['Precision']:.8f}"
)

print(
    f"Recall/Sensitivity: "
    f"{complete_oof_metrics['Recall/Sensitivity']:.8f}"
)

print(
    f"Specificity: "
    f"{complete_oof_metrics['Specificity']:.8f}"
)

print(
    f"F1: "
    f"{complete_oof_metrics['F1']:.8f}"
)

print("\nFold-level summary:")

print(
    f"Accuracy: "
    f"{mean_fold_accuracy:.8f} "
    f"± {sd_fold_accuracy:.8f}"
)

print(
    f"ROC-AUC: "
    f"{mean_fold_auc:.8f} "
    f"± {sd_fold_auc:.8f}"
)

print(
    f"F1: "
    f"{mean_fold_f1:.8f} "
    f"± {sd_fold_f1:.8f}"
)

print("\nSaved seed-42 reference:")

print(
    f"TN={saved_tn}, "
    f"FP={saved_fp}, "
    f"FN={saved_fn}, "
    f"TP={saved_tp}"
)

print(
    f"Pooled accuracy: "
    f"{saved_pooled_accuracy:.8f}"
)

print(
    f"Mean fold accuracy: "
    f"{saved_mean_fold_accuracy:.8f}"
)

print(
    f"Mean fold ROC-AUC: "
    f"{saved_mean_fold_auc:.8f}"
)

print("\nRerun comparison:")

print(
    "Confusion matrix exact match:",
    confusion_matrix_exact_match,
)

print(
    f"Pooled accuracy difference: "
    f"{accuracy_difference:+.8f}"
)

print(
    f"Mean fold accuracy difference: "
    f"{mean_fold_accuracy_difference:+.8f}"
)

print(
    f"Mean fold ROC-AUC difference: "
    f"{mean_fold_auc_difference:+.8f}"
)

print(
    "Meaningful accuracy discrepancy (>0.005):",
    meaningful_accuracy_discrepancy,
)

print(
    f"\nTotal runtime: "
    f"{total_runtime_minutes:.2f} minutes"
)

print("\nSaved OOF predictions:")
print(OOF_PATH)

print("\nSaved complete metrics:")
print(METRICS_PATH)

print("\nSaved per-fold metrics:")
print(PER_FOLD_PATH)

print("\nSaved fold comparison:")
print(FOLD_COMPARISON_PATH)

print("\nSaved fold-specific models:")
print(FOLD_MODEL_DIR)

print(
    "\nDo not generate the final TIFF figures until "
    "these results have been reviewed."
)